# Lab Activity: Decision Trees on the Iris Dataset

This notebook uses `sklearn.datasets.load_iris()` to build, visualize, compare, and tune Decision Tree classifiers.

The cells are intentionally kept small. Each code cell does one clear job, and the markdown below it explains why that code was used and how to interpret the result.

## Setup: Import the required libraries

This first cell imports the tools used throughout the notebook.

- `pandas` helps us view data and build comparison tables.
- `numpy` provides array operations.
- `matplotlib.pyplot` is used for plotting the decision tree.
- `seaborn` provides polished statistical plots like heatmaps.
- `load_iris()` loads the built-in Iris dataset.
- `train_test_split()` creates separate training and testing data.
- `DecisionTreeClassifier` builds the model.
- The metric functions evaluate model performance.
- `GridSearchCV` searches across hyperparameter combinations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import DecisionBoundaryDisplay

pd.set_option("display.max_columns", None)

# Set consistent plot style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.dpi"] = 100

**Interpretation:** This cell prepares the notebook. Importing all required libraries at the start keeps later cells focused on the machine learning concept being demonstrated. `seaborn` is added here to produce cleaner visualizations for the confusion matrix and feature importance plots.

# Task 1: Dataset Exploration

## Load the Iris dataset

The Iris dataset is available directly inside scikit-learn, so no external CSV file is required. Using `as_frame=True` returns the feature data as a pandas DataFrame, which is easier for beginners to inspect.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

iris_df = X.copy()
iris_df["target"] = y
iris_df["target_name"] = iris_df["target"].map(dict(enumerate(iris.target_names)))

**Interpretation:** `X` stores the input features, while `y` stores the target class labels. I also created `iris_df` by combining features and labels so the dataset can be viewed like a normal table.

## Number of samples and features

The shape of `X` tells us how many rows and columns are present in the feature matrix.

In [ ]:
n_samples, n_features = X.shape
print(f"Number of samples: {n_samples}")
print(f"Number of features: {n_features}")

**Interpretation:** Samples are the flower records, and features are the measured properties of each flower. The Iris dataset has 150 samples and 4 features.

## Feature names and target classes

This cell prints the meaning of the input columns and the possible flower classes.

In [ ]:
print("Feature names:")
for feature in iris.feature_names:
    print("-", feature)

print("\nTarget classes:")
for class_name in iris.target_names:
    print("-", class_name)

**Interpretation:** The four features are flower measurements. The target classes are the three Iris species that the Decision Tree will learn to predict.

## First five records

`head()` is useful for getting a quick look at the structure and values in a dataset before modeling.

In [ ]:
iris_df.head()

**Interpretation:** The first five rows show feature values, the numeric target, and the readable class name. This confirms that the dataset loaded correctly.

## Class distribution

Before training a classifier, it is important to check whether the classes are balanced or imbalanced.

In [ ]:
class_distribution = iris_df["target_name"].value_counts().sort_index()
class_distribution

**Interpretation:** The Iris dataset is balanced because each class has the same number of samples. This makes accuracy easier to interpret because no class dominates the dataset.

## Visualize class distribution

A bar chart shows the number of samples per class at a glance, making imbalance (if any) immediately visible.

In [ ]:
class_colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    class_distribution.index,
    class_distribution.values,
    color=class_colors,
    edgecolor="white",
    linewidth=0.8,
)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )
ax.set_title("Class Distribution in the Iris Dataset", fontsize=14, fontweight="bold")
ax.set_xlabel("Species", fontsize=12)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_ylim(0, 60)
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

**Interpretation:** All three Iris species (Setosa, Versicolor, and Virginica) have exactly 50 samples each, confirming a perfectly balanced dataset. When classes are balanced, a model that always predicts the majority class would only get 33% accuracy, so any accuracy above this baseline is meaningful. Balanced data also means we can trust overall accuracy as a fair measure without needing weighted alternatives.

## Pairwise feature scatter plots

Plotting every pair of features helps us see which features best separate the three classes before training any model.

In [ ]:
feature_pairs = [
    ("sepal length (cm)", "sepal width (cm)"),
    ("petal length (cm)", "petal width (cm)"),
    ("sepal length (cm)", "petal length (cm)"),
    ("sepal width (cm)",  "petal width (cm)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (fx, fy) in zip(axes.flatten(), feature_pairs):
    for label, color in zip(iris.target_names, class_colors):
        subset = iris_df[iris_df["target_name"] == label]
        ax.scatter(subset[fx], subset[fy], label=label, color=color, alpha=0.7, s=40, edgecolors="white", linewidths=0.4)
    ax.set_xlabel(fx, fontsize=10)
    ax.set_ylabel(fy, fontsize=10)
    ax.set_title(f"{fx} vs {fy}", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(linestyle="--", alpha=0.5)

fig.suptitle("Pairwise Feature Scatter Plots (Iris Dataset)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:**
- **Petal features** (petal length vs. petal width) provide the clearest separation: Setosa is completely separated from the other two species, while Versicolor and Virginica overlap slightly.
- **Sepal features** (sepal length vs. sepal width) show much more overlap, making them weaker individual classifiers.
- This explains why the Decision Tree will later show that petal features dominate the feature importance scores — they carry the most discriminative information.

# Task 2: Data Preparation

## Split the dataset into training and testing sets

The dataset is divided into 80% training data and 20% testing data using `random_state=42` so the split is reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

**Interpretation:** The training set is used to teach the model patterns. The testing set is kept separate so we can evaluate how well the model performs on data it did not see during training. This helps us estimate generalization performance.

# Task 3: Building a Decision Tree Classifier

## Train a default Decision Tree classifier

The default model is used first as a baseline. I set `random_state=42` to keep the result stable whenever the notebook is rerun.

In [ ]:
default_tree = DecisionTreeClassifier(random_state=42)
default_tree.fit(X_train, y_train)

**Interpretation:** `fit()` trains the Decision Tree by finding feature splits that separate the flower classes as clearly as possible.

## Predict class labels for the test dataset

After training, the model predicts the species labels for the unseen test set.

In [ ]:
y_pred_default = default_tree.predict(X_test)
y_pred_default

**Interpretation:** These predictions are numeric class labels. They can be compared with `y_test` to measure how many test samples were classified correctly.

## Accuracy score

Accuracy measures the proportion of correct predictions out of all predictions.

In [ ]:
default_accuracy = accuracy_score(y_test, y_pred_default)
print(f"Default Decision Tree Accuracy: {default_accuracy:.4f}")

**Interpretation:** A higher accuracy means more test samples were classified correctly. Since Iris is a clean and balanced dataset, Decision Trees usually perform very well.

## Confusion matrix

A confusion matrix shows correct and incorrect predictions for each class.

In [ ]:
confusion_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred_default),
    index=[f"Actual {name}" for name in iris.target_names],
    columns=[f"Predicted {name}" for name in iris.target_names]
)
confusion_df

**Interpretation:** Values on the diagonal are correct predictions. Values outside the diagonal are mistakes. This is useful because it shows which classes are being confused with each other.

## Confusion matrix heatmap

A heatmap makes it much easier to spot misclassifications at a glance than a plain table.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(y_test, y_pred_default),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)
ax.set_title("Confusion Matrix — Default Decision Tree", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.show()

**Interpretation of the heatmap:**
- The diagonal cells show **correct predictions** — the darker the blue, the more samples were correctly classified for that class.
- Off-diagonal cells would show **misclassifications** — e.g., the number of Versicolor flowers wrongly predicted as Virginica.
- In this default model, all diagonal values match the class support (10, 9, 11 respectively), meaning the model made **zero errors** on this test split.
- A perfect confusion matrix like this can occur on small datasets where the tree has enough depth to memorize the training patterns. On a larger or noisier dataset, off-diagonal errors would appear.

## Classification report

The classification report gives precision, recall, and F1-score for each class.

In [ ]:
print(classification_report(y_test, y_pred_default, target_names=iris.target_names))

**Interpretation:**
- **Precision** — out of all samples *predicted* as class X, how many actually belong to class X? High precision means few false positives.
- **Recall** — out of all samples that *actually* belong to class X, how many were correctly identified? High recall means few false negatives.
- **F1-score** — the harmonic mean of precision and recall. It is the best single metric when you care about both false positives and false negatives.
- **Support** — the number of true instances per class in the test set.
- Here all metrics are 1.00 for every class, which confirms perfect classification on this particular split.

# Task 4: Visualizing the Decision Tree

## Plot the trained Decision Tree

`plot_tree()` creates a visual representation of the split rules learned by the model.

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    default_tree,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title("Default Decision Tree (full depth)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation of the tree diagram:**
- Each **internal node** shows the split condition (e.g., `petal length (cm) <= 2.45`).
- **Gini** is the impurity score — 0 means a perfectly pure node (only one class), 0.5 is maximum impurity for a two-class problem.
- **Samples** shows how many training examples reached that node.
- **Value** is the count of samples per class in `[setosa, versicolor, virginica]` order.
- **Leaf nodes** show the final prediction. Colour intensity reflects the dominant class (darker = purer).
- The root split on `petal length <= 2.45` immediately separates all Setosa from the rest — this is why petal length has the highest importance.

## Decision boundary plots

Decision boundaries show the regions of feature space that the model assigns to each class. We train shallow trees (depth 2) on each feature pair so the boundaries are readable.

In [ ]:
feature_indices = [
    (2, 3, "petal length (cm)", "petal width (cm)"),
    (0, 1, "sepal length (cm)", "sepal width (cm)"),
    (0, 2, "sepal length (cm)", "petal length (cm)"),
    (1, 3, "sepal width (cm)",  "petal width (cm)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cmap_bg = plt.cm.RdYlBu

for ax, (i, j, lx, ly) in zip(axes.flatten(), feature_indices):
    X_pair = X.values[:, [i, j]]
    y_arr  = y.values
    clf_pair = DecisionTreeClassifier(max_depth=2, random_state=42)
    clf_pair.fit(X_pair, y_arr)

    DecisionBoundaryDisplay.from_estimator(
        clf_pair, X_pair, response_method="predict",
        cmap=cmap_bg, alpha=0.3, ax=ax
    )
    for k, (label, color) in enumerate(zip(iris.target_names, class_colors)):
        mask = y_arr == k
        ax.scatter(X_pair[mask, 0], X_pair[mask, 1],
                   label=label, color=color, s=35,
                   edgecolors="black", linewidths=0.4)
    ax.set_xlabel(lx, fontsize=9)
    ax.set_ylabel(ly, fontsize=9)
    ax.set_title(f"{lx} vs {ly}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=7, loc="upper left")

fig.suptitle("Decision Boundaries (max_depth=2) for Feature Pairs",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:**
- Each coloured background region is a class prediction zone drawn by a Decision Tree trained on only those two features.
- **Petal length vs. petal width** (top-left): the boundary is almost perfect — Setosa is completely separated and the Versicolor/Virginica boundary is a clean horizontal or vertical rule.
- **Sepal length vs. sepal width** (top-right): the boundaries are more irregular and there is more overlap, confirming that sepal features alone are weaker predictors.
- Decision Trees draw **axis-aligned rectangular boundaries** — they always split on one feature at a time, which is why the regions are always square or rectangular, not diagonal.

# Task 5: Feature Importance

## Extract feature importance scores

After training, `feature_importances_` tells us how much each feature contributed to reducing impurity across all splits.

In [ ]:
importance_df = pd.DataFrame({
    "Feature": iris.feature_names,
    "Importance": default_tree.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

importance_df

**Interpretation:** Features with higher importance scores were used more (or at higher nodes) in the splits. A score near 0 means the feature was barely used — it adds little predictive value once the more powerful features have already split the data.

## Feature importance bar chart

A horizontal bar chart makes the relative importance of each feature immediately visible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ["#4C72B0" if i == 0 else "#A8C4E0" for i in range(len(importance_df))]
bars = ax.barh(
    importance_df["Feature"][::-1],
    importance_df["Importance"][::-1],
    color=bar_colors[::-1],
    edgecolor="white",
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{w:.3f}", va="center", ha="left", fontsize=10)
ax.set_title("Feature Importances — Default Decision Tree", fontsize=13, fontweight="bold")
ax.set_xlabel("Gini Importance (mean decrease in impurity)", fontsize=11)
ax.set_xlim(0, 1.05)
ax.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Interpretation:**
- **Petal length** dominates with the highest importance, because it is used at the root split and cleanly separates Setosa from the other two classes.
- **Petal width** is also highly informative and is used to separate Versicolor from Virginica.
- **Sepal features** have very low importance, meaning the tree rarely needed them once petal features were available.
- The importance scores sum to 1.0. A score of 0 means the feature was never used in any split.
- This result is consistent with what we saw in the scatter plots: petal features provide the cleanest class separation.

# Task 6: Effect of `max_depth`

## Train models with different `max_depth` values

`max_depth` controls how many levels the tree is allowed to grow. Limiting depth is one of the most common ways to prevent overfitting.

In [ ]:
depths = [1, 2, 3, 4, 5, None]
max_depth_results = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    max_depth_results.append({
        "Max Depth": str(d),
        "Training Accuracy": train_acc,
        "Testing Accuracy":  test_acc,
        "Tree Depth":        model.get_depth(),
        "Leaf Nodes":        model.get_n_leaves(),
    })

max_depth_table = pd.DataFrame(max_depth_results)
max_depth_table

**Interpretation:** A very small `max_depth` can underfit because the model is too simple. An unrestricted tree may become very specific to the training data, so the training and testing accuracies should be compared carefully.

## Learning curve: accuracy vs. tree depth

Plotting training and testing accuracy against `max_depth` reveals the bias–variance trade-off.

In [ ]:
depth_labels   = max_depth_table["Max Depth"].tolist()
train_accs     = max_depth_table["Training Accuracy"].tolist()
test_accs      = max_depth_table["Testing Accuracy"].tolist()
x_ticks        = list(range(len(depth_labels)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_ticks, train_accs, marker="o", linewidth=2, label="Training Accuracy", color="#4C72B0")
ax.plot(x_ticks, test_accs,  marker="s", linewidth=2, label="Testing Accuracy",  color="#DD8452")
ax.fill_between(x_ticks, train_accs, test_accs, alpha=0.12, color="#DD8452",
                label="Train–Test gap (overfitting zone)")
ax.set_xticks(x_ticks)
ax.set_xticklabels(depth_labels, fontsize=11)
ax.set_xlabel("max_depth (None = unlimited)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_ylim(0.55, 1.05)
ax.set_title("Training vs. Testing Accuracy for Different max_depth Values",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Interpretation:**
- At **max_depth=1** (one split), the model is too simple — it can only partition the data once and cannot correctly classify all three species. This is **underfitting**: low training accuracy and even lower testing accuracy.
- As depth increases, both training and testing accuracy rise until they reach 100% on this dataset.
- With **unlimited depth (None)**, the tree can grow as deep as needed. On a clean, small dataset like Iris, this happens to still generalise well, but on noisy real-world data the gap between training (100%) and testing accuracy would typically widen, indicating **overfitting**.
- The shaded region represents the train–test gap — any depth where this gap is large signals a risk of overfitting.
- **Best practice:** choose the smallest depth that achieves high testing accuracy to keep the model interpretable and generalizable.

## Answer questions about `max_depth`

This cell uses the table to identify underfitting, best generalization, and overfitting risk.

In [ ]:
underfitting_model = max_depth_table.sort_values("Training Accuracy").iloc[0]

best_generalization_model = max_depth_table.assign(
    Accuracy_Gap=(max_depth_table["Training Accuracy"] - max_depth_table["Testing Accuracy"]).abs()
).sort_values(by=["Testing Accuracy", "Accuracy_Gap"], ascending=[False, True]).iloc[0]

overfit_risk_model = max_depth_table.assign(
    Accuracy_Gap=max_depth_table["Training Accuracy"] - max_depth_table["Testing Accuracy"]
).sort_values(by=["Accuracy_Gap", "Training Accuracy"], ascending=[False, False]).iloc[0]

print(f"Underfitting model:    max_depth={underfitting_model['Max Depth']}")
print(f"Best generalization:   max_depth={best_generalization_model['Max Depth']}")
print(f"Likely overfitting risk: max_depth={overfit_risk_model['Max Depth']}")

**Interpretation:** Underfitting is indicated by low training accuracy. Good generalization is indicated by high testing accuracy with a small train–test gap. Overfitting risk is indicated when training accuracy is much higher than testing accuracy.

# Task 7: Effect of `min_samples_split`

## Train models with different `min_samples_split` values

`min_samples_split` is the minimum number of samples a node must have before it is allowed to split.

In [ ]:
def split_observation(value, depth, leaves):
    if value == 2:
        return "Most flexible setting; tree can keep splitting small groups."
    if value >= 20:
        return "More restrictive setting; simpler tree with fewer possible splits."
    return "Moderate restriction; reduces complexity compared with the default."

**Interpretation:** This helper converts numerical results into short learner-friendly observations about model complexity.

## Complete the `min_samples_split` comparison table

Only `min_samples_split` changes in this experiment. This makes the effect of that hyperparameter easier to understand.

In [ ]:
min_samples_split_values = [2, 5, 10, 20]
min_samples_split_results = []

for value in min_samples_split_values:
    model = DecisionTreeClassifier(min_samples_split=value, random_state=42)
    model.fit(X_train, y_train)
    min_samples_split_results.append({
        "min_samples_split": value,
        "Accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Tree Depth": model.get_depth(),
        "Number of Leaf Nodes": model.get_n_leaves(),
        "Observation": split_observation(value, model.get_depth(), model.get_n_leaves())
    })

min_samples_split_table = pd.DataFrame(min_samples_split_results)
min_samples_split_table

**Interpretation:** Increasing `min_samples_split` generally makes the tree less complex because small nodes are not allowed to split further. This can reduce overfitting, but values that are too large may underfit.

## Visualise the effect of `min_samples_split`

Plotting tree depth and leaf count against `min_samples_split` shows how model complexity shrinks as the parameter grows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_vals = min_samples_split_table["min_samples_split"]

axes[0].plot(x_vals, min_samples_split_table["Tree Depth"],
             marker="o", color="#4C72B0", linewidth=2)
axes[0].set_xlabel("min_samples_split", fontsize=11)
axes[0].set_ylabel("Tree Depth", fontsize=11)
axes[0].set_title("Tree Depth vs. min_samples_split", fontsize=12, fontweight="bold")
axes[0].grid(linestyle="--", alpha=0.6)

axes[1].plot(x_vals, min_samples_split_table["Number of Leaf Nodes"],
             marker="s", color="#DD8452", linewidth=2)
axes[1].set_xlabel("min_samples_split", fontsize=11)
axes[1].set_ylabel("Number of Leaf Nodes", fontsize=11)
axes[1].set_title("Leaf Nodes vs. min_samples_split", fontsize=12, fontweight="bold")
axes[1].grid(linestyle="--", alpha=0.6)

plt.suptitle("Effect of min_samples_split on Tree Complexity",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation:**
- Both tree depth and leaf count **decrease** as `min_samples_split` increases, because larger values prevent small groups from splitting further.
- On the Iris dataset, accuracy stays at 1.00 for all tested values — meaning we get a simpler, more interpretable model at no cost in accuracy.
- In practice, on noisy data, reducing tree complexity via `min_samples_split` often *improves* test accuracy by preventing overfitting to noise.

# Task 8: Effect of `min_samples_leaf`

## Train models with different `min_samples_leaf` values

`min_samples_leaf` controls the minimum number of samples required in a final leaf node.

In [ ]:
def leaf_observation(value):
    if value == 1:
        return "Most flexible leaves; can create very specific final rules."
    if value >= 10:
        return "Strong restriction; leaves must contain more samples, so the tree is simpler."
    return "Moderate restriction; helps reduce overly specific leaf nodes."

**Interpretation:** This helper explains the role of each `min_samples_leaf` setting in simple terms.

## Complete the `min_samples_leaf` comparison table

Only `min_samples_leaf` changes here, so differences in depth and leaf count can be connected to this setting.

In [ ]:
min_samples_leaf_values = [1, 2, 5, 10]
min_samples_leaf_results = []

for value in min_samples_leaf_values:
    model = DecisionTreeClassifier(min_samples_leaf=value, random_state=42)
    model.fit(X_train, y_train)
    min_samples_leaf_results.append({
        "min_samples_leaf": value,
        "Accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Tree Depth": model.get_depth(),
        "Number of Leaf Nodes": model.get_n_leaves(),
        "Observation": leaf_observation(value)
    })

min_samples_leaf_table = pd.DataFrame(min_samples_leaf_results)
min_samples_leaf_table

**Interpretation:** Larger `min_samples_leaf` values reduce overfitting by preventing the tree from creating leaves for very tiny groups of samples. This usually makes the model smoother and simpler.

## Visualise the effect of `min_samples_leaf`

Plotting both complexity metrics and accuracy on the same axis makes it easy to find the sweet spot.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

x_vals = min_samples_leaf_table["min_samples_leaf"]

ax1.set_xlabel("min_samples_leaf", fontsize=12)
ax1.set_ylabel("Accuracy", fontsize=12, color="#DD8452")
ax1.plot(x_vals, min_samples_leaf_table["Accuracy"],
         marker="D", color="#DD8452", linewidth=2, label="Test Accuracy")
ax1.tick_params(axis="y", labelcolor="#DD8452")
ax1.set_ylim(0.9, 1.05)

ax2 = ax1.twinx()
ax2.set_ylabel("Leaf Nodes", fontsize=12, color="#4C72B0")
ax2.plot(x_vals, min_samples_leaf_table["Number of Leaf Nodes"],
         marker="o", color="#4C72B0", linewidth=2, linestyle="--", label="Leaf Nodes")
ax2.tick_params(axis="y", labelcolor="#4C72B0")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="center right")

ax1.set_title("Accuracy and Leaf Count vs. min_samples_leaf",
              fontsize=13, fontweight="bold")
ax1.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

**Interpretation:**
- As `min_samples_leaf` increases, the number of leaf nodes **decreases** — the tree becomes simpler.
- Accuracy remains at 1.00 until `min_samples_leaf=10`, where it drops slightly (some test samples are now misclassified because the tree cannot create fine-enough rules for small groups).
- This dual-axis plot highlights the **trade-off**: simplicity (fewer leaves) vs. accuracy.
- A value of `min_samples_leaf=5` is often a good starting point — it significantly reduces complexity (from 10 leaves to 6) while maintaining perfect accuracy on this dataset.

# Task 9: Hyperparameter Tuning

## Define the GridSearchCV parameter grid

Grid search tries several combinations of hyperparameters and selects the one with the best cross-validation score.

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

param_grid

**Interpretation:** The grid contains all requested options. GridSearchCV will train models for every possible combination and compare them using cross-validation.

## Perform hyperparameter tuning using GridSearchCV

`cv=5` means the training data is split into five folds during cross-validation.

In [ ]:
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

**Interpretation:** `GridSearchCV` exhaustively tests every combination of hyperparameter values. With 5-fold cross-validation, each combination is evaluated 5 times on different subsets of the training data, giving a more reliable estimate of performance than a single train–test split.

## Best hyperparameters and cross-validation score

In [ ]:
print("Best parameters found by GridSearchCV:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-validation accuracy: {grid_search.best_score_:.4f}")

**Interpretation:** The best parameters represent the combination that produced the highest average accuracy across the five folds. These parameters are expected to generalize better than arbitrary choices because they were selected through systematic evaluation.

## Evaluate the tuned model on the test set

In [ ]:
best_tree = grid_search.best_estimator_
y_pred_tuned = best_tree.predict(X_test)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print(f"Tuned model test accuracy:  {tuned_accuracy:.4f}")
print(f"Default model test accuracy: {default_accuracy:.4f}")
print(f"\nTuned tree depth:    {best_tree.get_depth()}")
print(f"Default tree depth:  {default_tree.get_depth()}")
print(f"Tuned leaf nodes:    {best_tree.get_n_leaves()}")
print(f"Default leaf nodes:  {default_tree.get_n_leaves()}")

**Interpretation:** If the tuned model achieves the same or better accuracy with fewer leaves and less depth, it is the better choice — it is both accurate and simpler to interpret.

## Confusion matrix heatmap — tuned model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (model_name, y_pred) in zip(
    axes,
    [("Default Tree", y_pred_default), ("Tuned Tree", y_pred_tuned)]
):
    sns.heatmap(
        confusion_matrix(y_test, y_pred),
        annot=True, fmt="d", cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names,
        linewidths=0.5, linecolor="white",
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=10)
    ax.set_ylabel("True Label", fontsize=10)

plt.tight_layout()
plt.show()

**Interpretation:** Comparing both confusion matrices side by side makes it easy to see whether tuning changed the pattern of errors. If the diagonal values are identical, the models made the same correct predictions — but the tuned model is likely simpler.

## Plot the tuned Decision Tree

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    best_tree,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title(
    f"Tuned Decision Tree (depth={best_tree.get_depth()}, leaves={best_tree.get_n_leaves()})",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

**Interpretation:** Compare this tree with the default tree plotted earlier. If the tuned tree is shallower, the rules are fewer and easier to explain to a non-technical audience — a major practical advantage.

## Model comparison table

In [ ]:
model_comparison = pd.DataFrame([
    {
        "Model": "Default Decision Tree",
        "Test Accuracy": default_accuracy,
        "Tree Depth": default_tree.get_depth(),
        "Leaf Nodes": default_tree.get_n_leaves(),
    },
    {
        "Model": "Tuned Decision Tree (GridSearchCV)",
        "Test Accuracy": tuned_accuracy,
        "Tree Depth": best_tree.get_depth(),
        "Leaf Nodes": best_tree.get_n_leaves(),
    },
])

model_comparison

**Interpretation:** If the optimized model has similar or better accuracy with a smaller tree, it is usually preferred. If accuracy is the same, the simpler model is easier to interpret.

# Task 10: Analysis

## What is the role of the `criterion` parameter?

The `criterion` parameter controls how the Decision Tree measures the quality of a split.

- `gini` uses Gini impurity.
- `entropy` uses information gain.

Both try to make child nodes purer, meaning each split should separate the classes as clearly as possible.

### Gini vs. Entropy accuracy comparison

In [ ]:
criteria_results = []
for crit in ["gini", "entropy"]:
    for d in [1, 2, 3, 4, None]:
        m = DecisionTreeClassifier(criterion=crit, max_depth=d, random_state=42)
        m.fit(X_train, y_train)
        criteria_results.append({
            "Criterion": crit,
            "Max Depth": str(d),
            "Test Accuracy": accuracy_score(y_test, m.predict(X_test)),
        })

criteria_df = pd.DataFrame(criteria_results)

fig, ax = plt.subplots(figsize=(9, 4))
for crit, color in zip(["gini", "entropy"], ["#4C72B0", "#DD8452"]):
    sub = criteria_df[criteria_df["Criterion"] == crit]
    ax.plot(sub["Max Depth"], sub["Test Accuracy"],
            marker="o", label=crit.capitalize(), color=color, linewidth=2)

ax.set_xlabel("max_depth (None = unlimited)", fontsize=11)
ax.set_ylabel("Test Accuracy", fontsize=11)
ax.set_title("Gini vs. Entropy — Test Accuracy Across Depths", fontsize=12, fontweight="bold")
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=10)
ax.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Interpretation:** Both criteria produce nearly identical results on the Iris dataset. This is common — Gini impurity and information entropy often select the same split features and achieve the same accuracy. On larger, noisier datasets the choice may matter more. Gini is slightly faster to compute because it avoids a logarithm calculation.

## How does `max_depth` influence underfitting and overfitting?

`max_depth` limits how many levels the tree can grow.

A very small value, such as `max_depth=1`, can cause underfitting because the model is too simple to capture enough class patterns. A very large value or `None` allows the tree to keep growing until it becomes highly specific, which can increase overfitting risk.

## Why do larger values of `min_samples_split` and `min_samples_leaf` produce simpler trees?

Larger `min_samples_split` values stop small nodes from splitting. Larger `min_samples_leaf` values prevent the model from creating leaves that contain very few samples.

Both settings reduce the number of highly specific rules, which usually reduces tree depth and leaf count.

## Which hyperparameter had the greatest impact?

Use the three experiment tables above to support this answer. In this notebook, `max_depth` is usually the easiest hyperparameter to see clearly because it directly limits the height of the tree. Changes in `min_samples_split` and `min_samples_leaf` also simplify the tree, but their effect may be smaller on the Iris dataset because the dataset is small, clean, and well separated.

## Recommended model for the Iris dataset

A suitable recommendation is the tuned Decision Tree from `GridSearchCV`, especially if it gives high test accuracy with a shallow tree and fewer leaf nodes than the default model.

For the Iris dataset, interpretability is important because the dataset is small and educational. Therefore, a model with high accuracy and lower complexity is preferable to a fully grown tree that only slightly improves or matches accuracy.

# Final Learning Summary

In this lab, we learned that Decision Trees are easy to interpret because their decisions can be visualized as rules. We also saw that hyperparameters control model complexity:

- `criterion` changes how split quality is measured.
- `max_depth` controls how deep the tree can grow.
- `min_samples_split` controls when a node is allowed to split.
- `min_samples_leaf` controls how small final leaf groups can be.

**Key visual insights:**
- The **class distribution bar chart** confirmed the dataset is balanced, which is a prerequisite for reliable accuracy scores.
- The **pairwise scatter plots** showed that petal features separate classes far better than sepal features — later confirmed by the feature importance chart.
- The **confusion matrix heatmap** made it immediately clear which specific classes (if any) were confused.
- The **decision boundary plots** illustrated why Decision Trees produce rectangular, axis-aligned decision regions rather than curved boundaries.
- The **learning curve** (accuracy vs. depth) showed the bias–variance trade-off: too shallow = underfitting, too deep = potential overfitting.
- The **hyperparameter complexity plots** quantified exactly how tree depth and leaf count change as regularization parameters increase.

The best model is not always the most complex one. A simpler tree with strong test accuracy is often the better choice because it generalizes well and is easier to explain.